#### Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 2. 

In [1]:
%load_ext autoreload
%autoreload 2

# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 2 - EBM-NLP

This dataset is provided at https://github.com/bepnye/EBM-NLP and a copy has been made available in this repository for convenience. The data will need to be unzipped:

In [2]:
import tarfile
import os

path_tofile = "./ebm_nlp_2_00.tar.gz"
extract_directory = os.path.dirname(path_tofile)

if tarfile.is_tarfile(path_tofile):
    with tarfile.open(path_tofile) as f:
        f.extractall(path=extract_directory)  # Extract all members from the archive to the current working directory


The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. So, let's deal with each type separately for now.

To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [3]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_2_00")
PHASES = ('starting_spans', 'hierarchical_labels')
ELEMENTS = ('participants', 'interventions', 'outcomes')
docs_dir = DATA_DIR / "documents"

def get_doc_ids(split="train", label_type="participants"):
    """ 
    split: 'train' or 'test' 
    """

    if split == "test":
        split = "test/gold"

    train_dir = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type  # assuming that the split is the same for all entity types, we can just look at one of them
        / split
    )
    
    doc_ids = [p.stem.split(".")[0] for p in train_dir.glob("*.AGGREGATED.ann")]
    #print(doc_ids)
    return sorted(doc_ids)

doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")


Number of documents in train split for participants: 4609
Number of documents in test split for participants: 189


Now, we can get the annotations for the first entity type:

In [4]:
def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test' 
    """
    if split == "test":
        split = "test/gold"

    ann_path = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / label_type / split/ f"{doc_id}.AGGREGATED.ann"
    
    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None
    
    with open(ann_path, "r", encoding="utf-8") as f:
        labels = [line.strip() for line in f]
    
    return labels

def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

participants_labels = load_labels(doc_ids_p, "participants", split="train")

print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

sample = 123
print("Document ID:", doc_ids_p[sample])
print(f"Participants label example for doc {doc_ids_p[sample]}:")
print(participants_labels[sample])

Length of participants_labels: 4609
Length of test_participants_labels: 189
Document ID: 10674680
Participants label example for doc 10674680:
['0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '4', '4', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '1', '1', '1', '1', '1', '1', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '

In [5]:
from itertools import chain
import numpy as np

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4']


Let's look at what labels there are for Participants. The code above shows there are four values: 0 corresponds to 'outside' but 1-4 all indicate tokens that form an entity span. Each number is a level in a hierarchy of specificity. To start with let's not worry about this 'hierarchy'. We can instead just turn the labels into simple BIO (Beginning of a span, Inside a span, and Outside a span) tags.

In [6]:
def hierarchical_to_bio(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B", "I").
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")

        prev = t
        

    return bio

def convert_all_labels_to_bio(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_bio(doc_labels)
    return labels

participants_labels = convert_all_labels_to_bio(participants_labels)
test_participants_labels = convert_all_labels_to_bio(test_participants_labels)

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['B' 'I' 'O']


So far, we've loaded the document IDs for participants and the corresponding labels. Now, let's load the documents themselves. They are already tokenised so that the labels match up with the tokens:

In [7]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)
# inspect a random element
print("Document ID:", doc_ids_p[sample])
print(f"Tokenised document example for doc {doc_ids_p[sample]}:")
print(participants_tokens[sample])


Document ID: 10674680
Tokenised document example for doc 10674680:
['Assessment', 'of', 'therapeutic', 'response', 'of', 'Plasmodium', 'falciparum', 'to', 'chloroquine', 'and', 'sulfadoxine-pyrimethamine', 'in', 'an', 'area', 'of', 'low', 'malaria', 'transmission', 'in', 'Colombia', '.', 'Although', 'chloroquine', '(', 'CQ', ')', 'resistance', 'was', 'first', 'reported', 'in', 'Colombia', 'in', '1961', 'and', 'sulfadoxine-pyrimethamine', '(', 'SP', ')', 'resistance', 'in', '1981', ',', 'the', 'frequency', 'of', 'treatment', 'failures', 'to', 'these', 'drugs', 'in', 'Colombia', 'is', 'unclear', '.', 'A', 'modified', 'World', 'Health', 'Organization', '14-day', 'in', 'vivo', 'drug', 'efficacy', 'test', 'for', 'uncomplicated', 'Plasmodium', 'falciparum', 'malaria', 'in', 'areas', 'with', 'intense', 'malaria', 'transmission', 'was', 'adapted', 'to', 'reflect', 'the', 'clinical', 'and', 'epidemiologic', 'features', 'of', 'a', 'low-intensity', 'malaria', 'transmission', 'area', 'in', 'the', 

### Interventions

In [8]:
doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

Number of documents in train split for interventions: 4746
Number of documents in test split for interventions: 187


In [9]:
interventions_labels = load_labels(doc_ids_i, "interventions", split="train")
print(f"Length of interventions_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_interventions_labels: {len(test_interventions_labels)}")

interventions_labels = convert_all_labels_to_bio(interventions_labels)
test_interventions_labels = convert_all_labels_to_bio(test_interventions_labels)


Length of interventions_labels: 4746
Length of test_interventions_labels: 187


### Outcomes

In [10]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Number of documents in train split for outcomes: 4681
Number of documents in test split for outcomes: 190


In [11]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")   
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_labels = convert_all_labels_to_bio(outcomes_labels)
test_outcomes_labels = convert_all_labels_to_bio(test_outcomes_labels)




Length of outcomes_labels: 4681
Length of test_outcomes_labels: 190


In [12]:
print(f"Intersection of train doc IDs across entity types: {len(set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o))}")
print(f"Intersection of test doc IDs across entity types: {len(set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o))}")
print(f"Documents that are different across entity types in train split: {len((set(doc_ids_p) | set(doc_ids_i) | set(doc_ids_o)) - (set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o)))}")
print(f"Documents that are different across entity types in test split: {len((set(test_doc_ids_p) | set(test_doc_ids_i) | set(test_doc_ids_o)) - (set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o)))}")


print(f"Test examples of the participants type that are in other entity types' training splits: {set(test_doc_ids_p) & (set(doc_ids_i) | set(doc_ids_o))}")
print(f"Test examples of the interventions type that are in other entity types' training splits: {set(test_doc_ids_i) & (set(doc_ids_p) | set(doc_ids_o))}")
print(f"Test examples of the outcomes type that are in other entity types' training splits: {set(test_doc_ids_o) & (set(doc_ids_p) | set(doc_ids_i))}")

Intersection of train doc IDs across entity types: 4457
Intersection of test doc IDs across entity types: 184
Documents that are different across entity types in train split: 344
Documents that are different across entity types in test split: 7
Test examples of the participants type that are in other entity types' training splits: set()
Test examples of the interventions type that are in other entity types' training splits: set()
Test examples of the outcomes type that are in other entity types' training splits: set()


In [13]:
import os, random
from glob import glob
from itertools import groupby, combinations
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, precision_recall_fscore_support


LABEL_DECODERS = { \
  PHASES[0] : { \
      'participants':  { 0: 'No Label', 1: 'p' },
      'interventions': { 0: 'No Label', 1: 'i' },
      'outcomes':      { 0: 'No Label', 1: 'o' }
    },
  PHASES[1]: { \
      'participants': { \
        0: 'No label',
        1: 'Age',
        2: 'Sex',
        3: 'Sample-size',
        4: 'Condition' },

      'interventions': { \
        0: 'No label',
        1: 'Surgical',
        2: 'Physical',
        3: 'Pharmacological',
        4: 'Educational',
        5: 'Psychological',
        6: 'Other',
        7: 'Control' },

      'outcomes': { \
        0: 'No label',
        1: 'Physical',
        2: 'Pain',
        3: 'Mortality',
        4: 'Adverse-effects',
        5: 'Mental',
        6: 'Other' }
    }
}

def rpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (s+' '*(n-len(s)))

def lpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (' '*(n-len(s))+s)

class Doc:
  def __init__(self, pmid, phase, element):
    with open(os.path.join(DATA_DIR, 'documents', '%s.txt' %pmid)) as fp:
      self.text = fp.read()
    with open(os.path.join(DATA_DIR, 'documents', '%s.tokens' %pmid)) as fp:
      self.tokens = fp.read().split('\n')
    self.pmid = pmid
    self.decoder = LABEL_DECODERS[phase][element]
    self.anns = {}

class Worker:
  def __init__(self, wid):
    self.wid = wid
    self.pmids = []

def get_pmids():
  doc_fnames = glob(os.path.join(DATA_DIR, 'documents', '*.text'))
  pmids = [os.path.basename(f).split('.')[0] for f in doc_fnames]
  return pmids

def read_anns(phase, element, ann_type = 'aggregated', model_phase = 'train'):
  workers = {}
  docs = {}

  fdir = os.path.join(DATA_DIR, 'annotations', ann_type, phase, element, model_phase)
  fnames = glob(os.path.join(fdir, '*.ann'))
  print('Found %d files in %s' %(len(fnames), fdir))

  for fname in fnames:
    labels = [int(i) for i in open(fname).read().strip().split('\n')]
    pmid, wid, f_ext = os.path.basename(fname).split('.')
    if pmid not in docs:
      docs[pmid] = Doc(pmid, phase, element)
    if wid not in workers:
      workers[wid] = Worker(wid)
    docs[pmid].anns[wid] = labels
    workers[wid].pmids.append(pmid)

  print('Loaded annotations for %d documents from %d worker%s' %(len(docs), len(workers), 's' if len(workers) != 1 else ''))
  return workers, docs

def print_token_labels(doc, width = 80): 
  t_str = '' 
  l_str = '' 
  for wid, labels in doc.anns.items():
    for t, l in zip(doc.tokens, labels):
      if l != 0:
        l_s = doc.decoder[l]
      else:
        l_s = ' '*len(t)
      slen = max(len(t), len(l_s))
      if len(t_str) + slen > width:
        if any([c != ' ' for c in l_str]):
          print(l_str)
        print(t_str)
        t_str = '' 
        l_str = '' 
      t_str += ' ' + rpad(t, slen)
      l_str += ' ' + rpad(l_s, slen)
    print(l_str)
    print(t_str)

def condense_labels(labels):
  groups = [(k, sum(1 for _ in g)) for k,g in groupby(labels)]
  spans = []
  i = 0
  for label, length in groups:
    if label != 0:
      spans.append((label, i, i+length))
    i += length
  return spans

def print_labeled_spans(doc):
  for wid, labels in doc.anns.items():
    label_spans = condense_labels(labels)
    print('Label spans for wid = %s' %wid)
    for label, token_i, token_f in label_spans:
      print('[%s]: %s ' %(doc.decoder[label], ' '.join(doc.tokens[token_i:token_f])))
    print()

def compute_worker_kappas(workers, docs):
  wids = sorted(workers.keys())
  worker_pairs = list(combinations(wids, 2))
  worker_kappas = [['' for _ in wids] for __ in wids]
  for (wid1, wid2) in worker_pairs:
    pmids = list(set(workers[wid1].pmids).intersection(workers[wid2].pmids))
    if len(pmids) > 0:
      l1 = sum([docs[pmid].anns[wid1] for pmid in pmids], [])
      l2 = sum([docs[pmid].anns[wid2] for pmid in pmids], [])
      kappa = cohen_kappa_score(l1, l2)
      idx1 = wids.index(wid1)
      idx2 = wids.index(wid2)
      worker_kappas[idx1][idx2] = kappa
      worker_kappas[idx2][idx1] = kappa

  print_matrix(worker_kappas, wids, 'Pairwise Cohen\'s Kappa')
  return worker_kappas

def print_matrix(matrix, row_names, title):
  row_names = row_names or ['' for row in matrix]
  title = title or 'Table'
  llen = max(map(len, row_names))
  print('%s:' %title)
  print('%s  %s' %(lpad('', llen), ' '.join([lpad(n, llen) for n in row_names])))
  for row,name in zip(matrix, row_names):
    print('%s: %s' %(lpad(name, llen), ' '.join([lpad(x if type(x) is str else '%.2f' %x, llen) for x in row])))

def add_dicts(d1, d2):
  d = d1.copy()
  d.update(d2)
  return d

def combine_model_phases(p1_data, p2_data):
  w1, d1 = p1_data
  w2, d2 = p2_data
  workers = add_dicts(w1, w2)
  docs = {}
  for pmid, d in d1.items():
    docs[pmid] = d
  for pmid, d in d2.items():
    if pmid not in docs:
      docs[pmid] = d
    else:
      docs[pmid].anns = dict(list(docs[pmid].anns.items()) + list(d.anns.items()))
  return workers, docs

def get_multiple_model_phases(phase, element, ann_type, phase1, phase2):
  p1 = read_anns(phase, element, ann_type, model_phase = phase1)
  p2 = read_anns(phase, element, ann_type, model_phase = phase2)
  return combine_model_phases(p1, p2)

def get_wid_color(wid):
  if wid == 'AGGREGATED':
    r = 0
    g = 150
    b = 50
  elif wid == 'UNION':
    r = 0
    g = 250
    b = 150
  else:
    r = int(random.random()*255)
    g = int(random.random()*126)
    b = 255
  color = '{:02x}{:02x}{:02x}'.format(r, g, b)
  return color

def write_brat_files(docs):
  fdir = 'brat/'
  while True:
    if not os.path.isdir(fdir):
      print('Please create the target directory: %s' %fdir)
      input('press [enter] when done  ')
    else:
      break
  wids = set()
  for pmid, doc in docs.items():
    offsets = [(0, len(doc.tokens[0]))]
    text = doc.tokens[0]
    for token in doc.tokens[1:]:
      spaced_token = ' ' + token
      offsets.append((len(text) + 1, len(text) + len(spaced_token)))
      text += spaced_token
      assert text[offsets[-1][0]:offsets[-1][1]] == token
    with open('%s/%s.txt' %(fdir, pmid), 'w') as fp:
      fp.write(text)
    with open('%s/%s.test.ann' %(fdir, pmid), 'w') as fp:
      tid = 0
      doc_wids = sorted(doc.anns.keys(), reverse = True)
      for wid in doc_wids:
        wids.add(wid)
        label_spans = condense_labels(doc.anns[wid])
        for label, token_i, token_f in label_spans:
          char_i = offsets[token_i][0]
          char_f = offsets[token_f-1][1]
          fp.write('T%d\t%s %d %d\t%s\n' %(tid, label, char_i, char_f, text[char_i:char_f]))
          tid += 1
  with open('%s/annotation.conf' %fdir, 'w') as fp:
    fp.write('[entities]\n\n')
    for wid in wids:
      wid = wid_translator.get(wid,wid)
      fp.write(wid+'\n')
    fp.write('[relations]\n\n')
    fp.write('<OVERLAP> Arg1:<ENTITY>, Arg2:<ENTITY>, <OVL-TYPE>:<ANY>\n\n')
    fp.write('[events]\n\n')
    fp.write('[attributes]\n\n')
  with open('%s/visual.conf' %fdir, 'w') as fp:
    fp.write('[drawing]\n\n')
    for wid in wids:
      color = get_wid_color(wid)
      wid = wid_translator.get(wid,wid)
      fp.write('%s bgColor:#%s\n' %(wid, color))

In [14]:
import os, random
from glob import glob
from itertools import groupby, combinations
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, precision_recall_fscore_support


LABEL_DECODERS = { \
  PHASES[0] : { \
      'participants':  { 0: 'No Label', 1: 'p' },
      'interventions': { 0: 'No Label', 1: 'i' },
      'outcomes':      { 0: 'No Label', 1: 'o' }
    },
  PHASES[1]: { \
      'participants': { \
        0: 'No label',
        1: 'Age',
        2: 'Sex',
        3: 'Sample-size',
        4: 'Condition' },

      'interventions': { \
        0: 'No label',
        1: 'Surgical',
        2: 'Physical',
        3: 'Pharmacological',
        4: 'Educational',
        5: 'Psychological',
        6: 'Other',
        7: 'Control' },

      'outcomes': { \
        0: 'No label',
        1: 'Physical',
        2: 'Pain',
        3: 'Mortality',
        4: 'Adverse-effects',
        5: 'Mental',
        6: 'Other' }
    }
}

def rpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (s+' '*(n-len(s)))

def lpad(inp, n, min_buf=0):
  s = str(inp)[:n - min_buf]
  return (' '*(n-len(s))+s)

class Doc:
  def __init__(self, pmid, phase, element):
    with open(os.path.join(DATA_DIR, 'documents', '%s.txt' %pmid)) as fp:
      self.text = fp.read()
    with open(os.path.join(DATA_DIR, 'documents', '%s.tokens' %pmid)) as fp:
      self.tokens = fp.read().split('\n')
    self.pmid = pmid
    self.decoder = LABEL_DECODERS[phase][element]
    self.anns = {}

class Worker:
  def __init__(self, wid):
    self.wid = wid
    self.pmids = []

def get_pmids():
  doc_fnames = glob(os.path.join(DATA_DIR, 'documents', '*.text'))
  pmids = [os.path.basename(f).split('.')[0] for f in doc_fnames]
  return pmids

def read_anns(phase, element, ann_type = 'aggregated', model_phase = 'train'):
  workers = {}
  docs = {}

  fdir = os.path.join(DATA_DIR, 'annotations', ann_type, phase, element, model_phase)
  fnames = glob(os.path.join(fdir, '*.ann'))
  print('Found %d files in %s' %(len(fnames), fdir))

  for fname in fnames:
    labels = [int(i) for i in open(fname).read().strip().split('\n')]
    pmid, wid, f_ext = os.path.basename(fname).split('.')
    if pmid not in docs:
      docs[pmid] = Doc(pmid, phase, element)
    if wid not in workers:
      workers[wid] = Worker(wid)
    docs[pmid].anns[wid] = labels
    workers[wid].pmids.append(pmid)

  print('Loaded annotations for %d documents from %d worker%s' %(len(docs), len(workers), 's' if len(workers) != 1 else ''))
  return workers, docs

def print_token_labels(doc, width = 80): 
  t_str = '' 
  l_str = '' 
  for wid, labels in doc.anns.items():
    for t, l in zip(doc.tokens, labels):
      if l != 0:
        l_s = doc.decoder[l]
      else:
        l_s = ' '*len(t)
      slen = max(len(t), len(l_s))
      if len(t_str) + slen > width:
        if any([c != ' ' for c in l_str]):
          print(l_str)
        print(t_str)
        t_str = '' 
        l_str = '' 
      t_str += ' ' + rpad(t, slen)
      l_str += ' ' + rpad(l_s, slen)
    print(l_str)
    print(t_str)

def condense_labels(labels):
  groups = [(k, sum(1 for _ in g)) for k,g in groupby(labels)]
  spans = []
  i = 0
  for label, length in groups:
    if label != 0:
      spans.append((label, i, i+length))
    i += length
  return spans

def print_labeled_spans(doc):
  for wid, labels in doc.anns.items():
    label_spans = condense_labels(labels)
    print('Label spans for wid = %s' %wid)
    for label, token_i, token_f in label_spans:
      print('[%s]: %s ' %(doc.decoder[label], ' '.join(doc.tokens[token_i:token_f])))
    print()

def compute_worker_kappas(workers, docs):
  wids = sorted(workers.keys())
  worker_pairs = list(combinations(wids, 2))
  worker_kappas = [['' for _ in wids] for __ in wids]
  for (wid1, wid2) in worker_pairs:
    pmids = list(set(workers[wid1].pmids).intersection(workers[wid2].pmids))
    if len(pmids) > 0:
      l1 = sum([docs[pmid].anns[wid1] for pmid in pmids], [])
      l2 = sum([docs[pmid].anns[wid2] for pmid in pmids], [])
      kappa = cohen_kappa_score(l1, l2)
      idx1 = wids.index(wid1)
      idx2 = wids.index(wid2)
      worker_kappas[idx1][idx2] = kappa
      worker_kappas[idx2][idx1] = kappa

  print_matrix(worker_kappas, wids, 'Pairwise Cohen\'s Kappa')
  return worker_kappas

def print_matrix(matrix, row_names, title):
  row_names = row_names or ['' for row in matrix]
  title = title or 'Table'
  llen = max(map(len, row_names))
  print('%s:' %title)
  print('%s  %s' %(lpad('', llen), ' '.join([lpad(n, llen) for n in row_names])))
  for row,name in zip(matrix, row_names):
    print('%s: %s' %(lpad(name, llen), ' '.join([lpad(x if type(x) is str else '%.2f' %x, llen) for x in row])))

def add_dicts(d1, d2):
  d = d1.copy()
  d.update(d2)
  return d

def combine_model_phases(p1_data, p2_data):
  w1, d1 = p1_data
  w2, d2 = p2_data
  workers = add_dicts(w1, w2)
  docs = {}
  for pmid, d in d1.items():
    docs[pmid] = d
  for pmid, d in d2.items():
    if pmid not in docs:
      docs[pmid] = d
    else:
      docs[pmid].anns = dict(list(docs[pmid].anns.items()) + list(d.anns.items()))
  return workers, docs

def get_multiple_model_phases(phase, element, ann_type, phase1, phase2):
  p1 = read_anns(phase, element, ann_type, model_phase = phase1)
  p2 = read_anns(phase, element, ann_type, model_phase = phase2)
  return combine_model_phases(p1, p2)

def get_wid_color(wid):
  if wid == 'AGGREGATED':
    r = 0
    g = 150
    b = 50
  elif wid == 'UNION':
    r = 0
    g = 250
    b = 150
  else:
    r = int(random.random()*255)
    g = int(random.random()*126)
    b = 255
  color = '{:02x}{:02x}{:02x}'.format(r, g, b)
  return color

def write_brat_files(docs):
  fdir = 'brat/'
  while True:
    if not os.path.isdir(fdir):
      print('Please create the target directory: %s' %fdir)
      input('press [enter] when done  ')
    else:
      break
  wids = set()
  for pmid, doc in docs.items():
    offsets = [(0, len(doc.tokens[0]))]
    text = doc.tokens[0]
    for token in doc.tokens[1:]:
      spaced_token = ' ' + token
      offsets.append((len(text) + 1, len(text) + len(spaced_token)))
      text += spaced_token
      assert text[offsets[-1][0]:offsets[-1][1]] == token
    with open('%s/%s.txt' %(fdir, pmid), 'w') as fp:
      fp.write(text)
    with open('%s/%s.test.ann' %(fdir, pmid), 'w') as fp:
      tid = 0
      doc_wids = sorted(doc.anns.keys(), reverse = True)
      for wid in doc_wids:
        wids.add(wid)
        label_spans = condense_labels(doc.anns[wid])
        for label, token_i, token_f in label_spans:
          char_i = offsets[token_i][0]
          char_f = offsets[token_f-1][1]
          fp.write('T%d\t%s %d %d\t%s\n' %(tid, label, char_i, char_f, text[char_i:char_f]))
          tid += 1
  with open('%s/annotation.conf' %fdir, 'w') as fp:
    fp.write('[entities]\n\n')
    for wid in wids:
      wid = wid_translator.get(wid,wid)
      fp.write(wid+'\n')
    fp.write('[relations]\n\n')
    fp.write('<OVERLAP> Arg1:<ENTITY>, Arg2:<ENTITY>, <OVL-TYPE>:<ANY>\n\n')
    fp.write('[events]\n\n')
    fp.write('[attributes]\n\n')
  with open('%s/visual.conf' %fdir, 'w') as fp:
    fp.write('[drawing]\n\n')
    for wid in wids:
      color = get_wid_color(wid)
      wid = wid_translator.get(wid,wid)
      fp.write('%s bgColor:#%s\n' %(wid, color))

In [15]:
all_sentences = []       
all_true_labels = []     

def extract_from_docs(docs, element):
    sentences = []
    true_labels = []
    for doc in docs.values():
        tokens = doc.tokens
        labels = list(doc.anns.values())[0] 
        if any(l != 0 for l in labels):
            sentence_text = " ".join(tokens)
            sentences.append(sentence_text)        
            true_labels.append(element)  
    return sentences, true_labels


_, p_docs = read_anns('starting_spans', 'participants', ann_type='aggregated', model_phase='train')
_, i_docs = read_anns('starting_spans', 'interventions', ann_type='aggregated', model_phase='train')
_, o_docs = read_anns('starting_spans', 'outcomes', ann_type='aggregated', model_phase='train')

p_sentences, p_labels = extract_from_docs(p_docs, 'participants')
i_sentences, i_labels = extract_from_docs(i_docs, 'interventions')
o_sentences, o_labels = extract_from_docs(o_docs, 'outcomes')

all_sentences = p_sentences + i_sentences + o_sentences
all_true_labels = p_labels + i_labels + o_labels

print(f"\n sum of sentencees: {len(all_sentences)} ，sum of labels:{len(all_true_labels)} ")


Found 4792 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/participants/train
Loaded annotations for 4792 documents from 1 worker
Found 4782 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/interventions/train
Loaded annotations for 4782 documents from 1 worker
Found 4670 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/outcomes/train
Loaded annotations for 4670 documents from 1 worker

 sum of sentencees: 14244 ，sum of labels:14244 


In [16]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModel
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", use_safetensors = True)
sentence_embeddings = []
with torch.no_grad():
    for text in all_sentences:
        model_input = tokenizer(text, padding=True, max_length=128,truncation=True, return_tensors="pt")  
        output = model(**model_input)
        hidden_states = output['last_hidden_state']
        cls_emb = hidden_states[0][0].detach().numpy()
        sentence_embeddings.append(cls_emb)

In [17]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
k = 3  
kmeans = KMeans(n_clusters=k, random_state=42)
embeddings_matrix = np.array(sentence_embeddings)
labels = kmeans.fit_predict(embeddings_matrix)
cross_df = pd.DataFrame({
    'KMeans_classification': labels, 
    'Annotations': all_true_labels
})
crosstab_result = pd.crosstab(cross_df['KMeans_classification'], cross_df['Annotations'])

print(crosstab_result)

Annotations            interventions  outcomes  participants
KMeans_classification                                       
0                                775       750           778
1                               3529      3453          3536
2                                478       467           478


In [25]:
#task 2
# extract  entity span
import random

#fixed 
#for i,pmid in enumerate(i_random_pimds):
    #doc = i_docs[pmid]
    #print(f"NO.{i+1}, ID: {pmid}")
    #print_labeled_spans(doc)
# Window = 4
# random.seed(42)
# i_random_pmids = random.sample(list(i_docs.keys()), 15)
# for i,pmid in enumerate(i_random_pmids):
#     doc = i_docs[pmid]
#     labels = list(doc.anns.values())[0]
#     label_spans = condense_labels(labels)
#     if not label_spans:
#         continue
#     print(f"NO.{i+1}, ID: {pmid}")
#     for label, token_i, token_f in label_spans:
#         l_span = " ".join(doc.tokens[max(0, token_i - Window) : token_i])
#         entity_span   = " ".join(doc.tokens[token_i : token_f])
#         r_span= " ".join(doc.tokens[token_f : min(len(doc.tokens), token_f + Window)])
#         print(f"[{doc.decoder[label]}]: ... {l_span} 【 {entity_span} 】 {r_span} ...")
#     print()
# def get_span_sentences(docs, element, num_samples = 15, window = 10, seed = 42):
#     results = []
#     random.seed(seed)
#     num_actual_samples = min(num_samples, len(docs))
#     random_pmids = random.sample(list(docs.keys()), num_actual_samples)
#     for i,pmid in enumerate(random_pmids):
#         doc = docs[pmid]
#         labels = list(doc.anns.values())[0]
#         label_spans = condense_labels(labels)
#         if not label_spans:
#             continue
#         for label, token_i, token_f in label_spans:
#             l_span = " ".join(doc.tokens[max(0, token_i - window) : token_i])
#             entity_span   = " ".join(doc.tokens[token_i : token_f])
#             r_span= " ".join(doc.tokens[token_f : min(len(doc.tokens), token_f + window)])
#             entitre_span = f"[{doc.decoder[label]}]: ... {l_span} 【 {entity_span} 】 {r_span} ..."
#             results.append(
#                 {
#                     "pmid": pmid,
#                     "span": entitre_span
#                 }
#             )
#     return results

def get_span_sentence(doc, window = 10):
    result = []
    labels = list(doc.anns.values())[0]
    label_spans = condense_labels(labels)
    if not label_spans:
        return result
    for label, token_i, token_f in label_spans:
        l_span = " ".join(doc.tokens[max(0, token_i - window) : token_i])
        entity_span   = " ".join(doc.tokens[token_i : token_f])
        r_span= " ".join(doc.tokens[token_f : min(len(doc.tokens), token_f + window)])
        entitre_span = f"[{doc.decoder[label]}]: ... {l_span} 【 {entity_span} 】 {r_span} ..."
        result.append(
            {
                "pmid": doc.pmid,
                "span": entitre_span
            }
        )
    return result

def get_span_sentences(docs, element, num_samples = 15, window = 10, seed = 42):
    results = []
    random.seed(seed)
    num_actual_samples = min(num_samples, len(docs))
    random_pmids = random.sample(list(docs.keys()), num_actual_samples)
    for pmid in random_pmids:
        doc = docs[pmid]
        doc_spans = get_span_sentence(doc, window=window)
        results.extend(doc_spans)
    return results

p_raw_spans = get_span_sentences(p_docs, 'participants')
for i, result in enumerate(p_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()
print("-" * 150)

i_raw_spans = get_span_sentences(i_docs, 'interventions')
for i, result in enumerate(i_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()
print("-" * 150)

o_raw_spans = get_span_sentences(o_docs, 'outcomes')
for i, result in enumerate(o_raw_spans):
    print(f"NO.{i+1}, ID: {result['pmid']}")
    print(f"{result['span']}")
    print()



NO.1, ID: 10889149
[p]: ... Atrophy and 【 intestinal metaplasia 】 one year after cure of H. pylori infection : a ...

NO.2, ID: 10889149
[p]: ... Atrophy and intestinal metaplasia one year after cure of 【 H. pylori infection : 】 a prospective , randomized study . BACKGROUND & AIMS Helicobacter ...

NO.3, ID: 10889149
[p]: ... course of premalignant histologic changes in the stomach . METHODS 【 Volunteers from the Yantai County in China underwent upper endoscopy with biopsy specimens obtained from the antrum and corpus . H. pylori-infected subjects 】 were randomized to receive either a 1-week course of omeprazole ...

NO.4, ID: 10889149
[p]: ... year , endoscopies with biopsies were repeated . RESULTS A 【 total of 587 H. pylori-infected subjects were randomized to OAC 】 ( n = 295 ) and placebo ( n = ...

NO.5, ID: 10889149
[p]: ... subjects assigned to OAC . In the placebo group , 【 245 patients remained H. pylori infected . 】 Analysis of paired samples obtained from the same patients s

In [19]:
import re
def extract_features_p(text):
    all_matches = []
    p_sample = re.findall(r'\d+\s+\b(?:patients|men|women|males?|females?)\b', text, re.IGNORECASE) 
    all_matches.extend(p_sample)
   
    # p_condition = re.findall(r'(?:(?:including|with|without)\s+)?(?:intestinal metaplasia|H\. pylori infection|\bOAC\b|allogenic bone marrow|rheumatology|non-urgent rheumatology|hypertension abundant phlegm-heat syndrome|\bQRHT\b|stage II/III colon cancer|ischemic heart disease)',
    #                          text, 
    #                          re.IGNORECASE)
    
    p_condition = re.findall(r'\b(?:patients|subjects|men|women|males?|females?)\s+(?:including|with|without)\s+[^\.,;]+',
                             text,
                             re.IGNORECASE)
                             
    all_matches.extend(p_condition)
        
    p_age = re.findall(r'\b(?:patients|men|women|male|female)\s*[≥≤><=]+\s*\d+\s*years?\b', 
                       text, 
                       re.IGNORECASE)
    all_matches.extend(p_age)
    return list(set(all_matches))


In [20]:
def extract_features_i(text):
 
    all_matches = []
    
    i_control = re.findall(r'\b(?:placebo|controlled)\b', text, re.IGNORECASE)
    all_matches.extend(i_control)
    
    i_drug = re.findall(r'\b\d+(?:\.\d+)?\s*(?:mg|ml)\b|\b\w+\s+(?:ointment|cream)\b', text, re.IGNORECASE)
    all_matches.extend(i_drug)
    
    i_surgical = re.findall(r'\b\w+\s+(?: therapy|resection|morcellation)\b', text, re.IGNORECASE)
    all_matches.extend(i_surgical)
    
    i_physical = re.findall(r'\b\w+\s+(?:ultrasound|devices?|monitoring)\b', text, re.IGNORECASE)
    all_matches.extend(i_physical)
    
    i_psycho = re.findall(r'\b\w+\s+(?:music|guided imagery|usual care)\b', text, re.IGNORECASE)
    all_matches.extend(i_psycho)
    return list(set(all_matches))

In [21]:
def extract_features_o(text):
    all_matches = []
    o_adverse = re.findall(r'\b(?:side effects?|adverse effects?|comlications?)\b', text, re.IGNORECASE)
    all_matches.extend(o_adverse)
    
    o_mental = re.findall(r'\b(?:cognitive|mental|agitation|distress|anxiety|depression)\b', text, re.IGNORECASE)
    all_matches.extend(o_mental)
    
    o_physical = re.findall(r'\b(?:\w+\s+)?(?:blood pressure|rate|blood flow|index|levels?|scores?)\b', text, re.IGNORECASE)
    all_matches.extend(o_physical)
    return list(set(all_matches))

In [22]:
import pandas as pd 

def get_pipeline(doc_ids, doc_tokens_list):
    final_results = []
    for pmid, tokens in zip(doc_ids, doc_tokens_list):
        full_text = " ".join(tokens).lower()
        p_results = extract_features_p(full_text)
        i_results = extract_features_i(full_text)
        o_results = extract_features_o(full_text)
        data = {
            "Document_ID": pmid,
            "Participants": " \n ".join(p_results) if len(p_results) > 0 else "Not Found",
            "Interventions": " \n ".join(i_results) if len(i_results) > 0 else "Not Found",
            "Outcomes": " \n ".join(o_results) if len(o_results) > 0 else "Not Found"
        } 
        final_results.append(data)    
    return final_results

In [23]:
# final_resuts_p = get_pipeline(test_doc_ids_p, test_participants_tokens)
# df_p = pd.DataFrame(final_resuts_p)
# print(df_p.head(10))

# final_resuts_i = get_pipeline(test_doc_ids_i, test_interventions_tokens)
# df_i = pd.DataFrame(final_resuts_i)
# print(df_i.head(10))

# final_resuts_o = get_pipeline(test_doc_ids_o, test_outcomes_tokens)
# df_o = pd.DataFrame(final_resuts_o)
# print(df_o.head(10))
all_test_ids = list(set(test_doc_ids_p + test_doc_ids_i + test_doc_ids_o))
all_test_tokens = load_documents(all_test_ids)
final_resuts = get_pipeline(all_test_ids, all_test_tokens)
df_final = pd.DataFrame(final_resuts)
print(df_final.head(10))

  Document_ID                                       Participants  \
0    21170734                                          Not Found   
1     8519720                                        20 patients   
2     9481998                                          Not Found   
3    17616069  women with clinical and lab signs of postparta...   
4    22646975          patients with inflammatory bowel disease    
5     6951573                                          Not Found   
6    24803369                                          Not Found   
7    18229990                                          Not Found   
8     7814711                                          Not Found   
9    22881991                                          Not Found   

                                       Interventions  \
0                         500 mg \n 700 mg \n 000 mg   
1     0.5 mg \n 16.7 mg \n lung resection \n 13.2 mg   
2  placebo \n controlled \n placebo cream \n of u...   
3                          

In [24]:
def get_predictions_label(tokens, predict_spans):
    predictions_label = [0] * len(tokens)
    if not predict_spans or predict_spans == ["Not Found"]:
        return predictions_label
    tokens = [t.lower() for t in tokens]
    for span in predict_spans:
        predict_tokens = span.lower().split()
        pre_tokens_len = len(predict_tokens)
        if pre_tokens_len == 0: 
            continue
        for i in range(len(tokens) - pre_tokens_len + 1):
            if tokens[i : i + pre_tokens_len] == predict_tokens:
                for j in range(i, i + pre_tokens_len):
                    predictions_label[j] = 1
    return predictions_label

def evaluate_predictions(test_docs, extract_features_func, element):
    true_labels = []
    pred_labels = []
    for pmid, doc in test_docs.items():
        true_label = list(doc.anns.values())[0]
        true_labels.extend(true_label)
        full_text = " ".join(doc.tokens).lower()
        predicted_spans = extract_features_func(full_text)
        pred_label = get_predictions_label(doc.tokens, predicted_spans)
        pred_labels.extend(pred_label)    
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average='binary', zero_division = 0)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f} ")


_, p_test_docs = read_anns('starting_spans', 'participants', ann_type='aggregated', model_phase='test/gold')
_, i_test_docs = read_anns('starting_spans', 'interventions', ann_type='aggregated', model_phase='test/gold')
_, o_test_docs = read_anns('starting_spans', 'outcomes', ann_type='aggregated', model_phase='test/gold')


evaluate_predictions(p_test_docs, extract_features_p, 'participants')
evaluate_predictions(i_test_docs, extract_features_i, 'interventions')
evaluate_predictions(o_test_docs, extract_features_o, 'outcomes')

Found 189 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/participants/test/gold
Loaded annotations for 189 documents from 1 worker
Found 188 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/interventions/test/gold
Loaded annotations for 188 documents from 1 worker
Found 190 files in ebm_nlp_2_00/annotations/aggregated/starting_spans/outcomes/test/gold
Loaded annotations for 190 documents from 1 worker
Precision: 0.5668 | Recall: 0.1919 | F1-Score: 0.2867 
Precision: 0.2937 | Recall: 0.0294 | F1-Score: 0.0535 
Precision: 0.6696 | Recall: 0.0468 | F1-Score: 0.0876 
